In [ ]:
# ruff: noqa: E402

# Illustrious XL v2.0 style LoRA training — Kaggle

In Kaggle Notebook settings, select **Accelerator: GPU**. Add these private
Kaggle Datasets as Inputs before running:

- the Shikishi dataset archive, mounted at `/kaggle/input/shikishi-ixy-style`;
- optionally, Illustrious-XL-v2.0.safetensors, mounted at
  `/kaggle/input/illustrious-xl-v20`. If omitted, enable Internet and the
  notebook downloads the public model itself.

To resume after a prior run, add the previous Notebook output (or an exported
state Dataset) as another Input. This notebook finds its newest valid state.

In [ ]:
from pathlib import Path

DATASET_NAME = "ixy_style"
OUTPUT_NAME = DATASET_NAME
INPUT_ROOT = Path("/kaggle/input")
# Kaggle may mount account-owned Dataset inputs under either /kaggle/input or
# /kaggle/input/datasets/<account>.  Support both layouts.
ACCOUNT_INPUT_ROOT = INPUT_ROOT / "datasets" / "palladiumailab"
DATASET_INPUT_DIR = next(
    (
        path
        for path in (
            INPUT_ROOT / "shikishi-ixy-style-prepared",
            ACCOUNT_INPUT_ROOT / "shikishi-ixy-style-prepared",
        )
        if path.is_dir()
    ),
    INPUT_ROOT / "shikishi-ixy-style-prepared",
)
MODEL_INPUT_DIR = next(
    (
        path
        for path in (INPUT_ROOT / "illustrious-xl-v20", ACCOUNT_INPUT_ROOT / "illustrious-xl-v20")
        if path.is_dir()
    ),
    INPUT_ROOT / "illustrious-xl-v20",
)
DATASET_ARCHIVE = DATASET_INPUT_DIR / f"{DATASET_NAME}-style-lora.zip"
INPUT_BASE_MODEL = MODEL_INPUT_DIR / "Illustrious-XL-v2.0.safetensors"
WORK_ROOT = Path("/kaggle/working")
BASE_MODEL = (
    INPUT_BASE_MODEL
    if INPUT_BASE_MODEL.is_file()
    else WORK_ROOT / "Illustrious-XL-v2.0.safetensors"
)
OUTPUT_DIR = WORK_ROOT / "outputs" / OUTPUT_NAME
LOG_DIR = WORK_ROOT / "logs"

SD_SCRIPTS_REF = "v0.9.1"
SD_SCRIPTS_COMMIT = "8f4ee8fc343b047965cd8976fca65c3a35b7593a"

In [ ]:
import os
import shutil
import sys

import torch


def require(condition: bool, message: str) -> None:
    if not condition:
        raise RuntimeError(message)


require(
    torch.cuda.is_available(),
    "GPU が見つかりません。Kaggle Settings で Accelerator を GPU に変更してください。",
)
gpu = torch.cuda.get_device_properties(0)
vram_gib = gpu.total_memory / 1024**3
gpu_capability = torch.cuda.get_device_capability(0)
free_gib = shutil.disk_usage(WORK_ROOT).free / 1024**3
require(
    vram_gib >= 14,
    f"GPU メモリが不足しています: {gpu.name} ({vram_gib:.1f} GiB)。P100/T4 以上を選んでください。",
)
require(
    gpu_capability >= (7, 0),
    f"GPU 世代が非対応です: {gpu.name} (sm_{gpu_capability[0]}{gpu_capability[1]})。"
    "Kaggle Settings で GPU T4 x2 を選んでください。",
)
require(free_gib >= 15, f"Kaggle作業領域の空きが不足しています: {free_gib:.1f} GiB")
MOUNTED_DATASET_ROOT = DATASET_INPUT_DIR / f"1_{DATASET_NAME}"
require(
    DATASET_ARCHIVE.is_file()
    or any(DATASET_INPUT_DIR.glob(f"{DATASET_ARCHIVE.name}.part*"))
    or MOUNTED_DATASET_ROOT.is_dir(),
    f"データセットInputが見つかりません: {DATASET_ARCHIVE}、分割ZIP、または {MOUNTED_DATASET_ROOT}",
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
print(f"GPU: {gpu.name} ({vram_gib:.1f} GiB)")
print(f"GPU capability: sm_{gpu_capability[0]}{gpu_capability[1]}")
print(f"Working disk free: {free_gib:.1f} GiB")

In [ ]:
import shlex
import subprocess


def run(label: str, *command: str) -> None:
    log_path = LOG_DIR / f"{DATASET_NAME}-{label}.log"
    print("+", shlex.join(command))
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
        return_code = process.wait()
    if return_code:
        raise RuntimeError(f"{label} が失敗しました (exit {return_code})。ログ: {log_path}")


repository = WORK_ROOT / "sd-scripts"
if repository.is_dir():
    completed = subprocess.run(
        ("git", "-C", str(repository), "rev-parse", "HEAD"), capture_output=True, text=True
    )
    if completed.returncode or completed.stdout.strip() != SD_SCRIPTS_COMMIT:
        shutil.rmtree(repository)
if not repository.is_dir():
    run(
        "clone-sd-scripts",
        "git",
        "clone",
        "--depth",
        "1",
        "--branch",
        SD_SCRIPTS_REF,
        "https://github.com/kohya-ss/sd-scripts.git",
        str(repository),
    )

actual_commit = subprocess.check_output(
    ("git", "-C", str(repository), "rev-parse", "HEAD"), text=True
).strip()
require(actual_commit == SD_SCRIPTS_COMMIT, f"sd-scripts の版が想定外です: {actual_commit}")
os.chdir(repository)
run(
    "install-sd-scripts",
    sys.executable,
    "-m",
    "pip",
    "install",
    "--quiet",
    "--disable-pip-version-check",
    "-r",
    "requirements.txt",
)
# Kaggle Python 3.12 image ships NumPy 2, while this stable sd-scripts stack
# imports TensorFlow binaries built against NumPy 1.x.
run(
    "install-numpy-compat",
    sys.executable,
    "-m",
    "pip",
    "install",
    "--quiet",
    "--disable-pip-version-check",
    "--force-reinstall",
    "numpy<2",
)
run("configure-accelerate", "accelerate", "config", "default", "--mixed_precision", "fp16")
run("verify-trainer", sys.executable, "sdxl_train_network.py", "--help")

if not BASE_MODEL.is_file():
    from huggingface_hub import hf_hub_download

    hf_hub_download(
        repo_id="OnomaAIResearch/Illustrious-XL-v2.0",
        filename=BASE_MODEL.name,
        local_dir=BASE_MODEL.parent,
    )
require(
    BASE_MODEL.is_file() and BASE_MODEL.stat().st_size > 6 * 1024**3,
    f"モデルが見つかりません、または不完全です: {BASE_MODEL}。"
    "Internetを有効にするか、モデルInputを追加してください。",
)

In [ ]:
import re
import zipfile

if MOUNTED_DATASET_ROOT.is_dir():
    # Kaggle's Dataset UI expands uploaded ZIPs into a read-only directory.
    # Reuse it directly instead of copying roughly 500 MB into /kaggle/working.
    EXTRACTED_ROOT = DATASET_INPUT_DIR
    print(f"Dataset source: mounted directory ({EXTRACTED_ROOT})")
else:
    part_pattern = re.compile(re.escape(DATASET_ARCHIVE.name) + r"\.part(\d+)$")
    parts = sorted(
        (
            (int(match.group(1)), path)
            for path in DATASET_INPUT_DIR.iterdir()
            if (match := part_pattern.match(path.name))
        ),
        key=lambda item: item[0],
    )
    if parts:
        part_numbers = [number for number, _ in parts]
        require(
            part_numbers == list(range(1, len(parts) + 1)),
            f"分割ZIPの番号が連続していません: {part_numbers}",
        )
        source_archive = WORK_ROOT / DATASET_ARCHIVE.name
        temporary_archive = source_archive.with_suffix(".partial")
        with temporary_archive.open("wb") as destination:
            for _, part in parts:
                require(part.stat().st_size > 0, f"空の分割ZIPがあります: {part.name}")
                with part.open("rb") as source:
                    shutil.copyfileobj(source, destination, length=16 * 1024 * 1024)
        temporary_archive.replace(source_archive)
    else:
        source_archive = DATASET_ARCHIVE

    with zipfile.ZipFile(source_archive) as archive:
        bad_member = next(
            (
                member.filename
                for member in archive.infolist()
                if Path(member.filename).is_absolute() or ".." in Path(member.filename).parts
            ),
            None,
        )
        require(bad_member is None, f"危険なZIPパスを検出しました: {bad_member}")
        corrupt_member = archive.testzip()
        require(corrupt_member is None, f"データセットZIPが壊れています: {corrupt_member}")
        EXTRACTED_ROOT = WORK_ROOT / "datasets" / f"{DATASET_NAME}_prepared"
        shutil.rmtree(EXTRACTED_ROOT, ignore_errors=True)
        archive.extractall(EXTRACTED_ROOT)
TRAIN_DATA_DIR = EXTRACTED_ROOT / f"1_{DATASET_NAME}"
VALIDATION_DATA_DIR = EXTRACTED_ROOT / "validation"
image_extensions = {".jpg", ".jpeg", ".png", ".webp"}
images = [path for path in TRAIN_DATA_DIR.iterdir() if path.suffix.lower() in image_extensions]
validation_images = [
    path for path in VALIDATION_DATA_DIR.iterdir() if path.suffix.lower() in image_extensions
] if VALIDATION_DATA_DIR.is_dir() else []
missing_captions = [
    path.name
    for path in images
    if not path.with_suffix(".txt").is_file()
    or not path.with_suffix(".txt").read_text(encoding="utf-8").strip()
]
require(images, f"画像が見つかりません: {TRAIN_DATA_DIR}")
require(
    not missing_captions, f"タグ .txt がない画像があります（先頭10件）: {missing_captions[:10]}"
)
require(TRAIN_DATA_DIR.is_dir(), "学習用フォルダ 1_<dataset_name> がありません。")
print(f"Dataset ready: {len(images)} train images, {len(validation_images)} validation images")

In [ ]:
os.chdir(repository)
state_pattern = re.compile(rf"^{re.escape(OUTPUT_NAME)}-step(\d+)-state$")
saved_states = sorted(
    (
        (int(match.group(1)), path)
        for input_path in INPUT_ROOT.iterdir()
        for path in input_path.rglob("*-state")
        if path.is_dir()
        and (match := state_pattern.match(path.name))
        and (path / "optimizer.bin").is_file()
    ),
    reverse=True,
)
resume_state = saved_states[0][1] if saved_states else None
if resume_state is None:
    print("Training mode: new run")
else:
    print(f"Training mode: resume from step {saved_states[0][0]} ({resume_state})")

train_command = (
    "accelerate",
    "launch",
    "--num_processes=1",
    "--num_machines=1",
    "--num_cpu_threads_per_process=2",
    "sdxl_train_network.py",
    f"--pretrained_model_name_or_path={BASE_MODEL}",
    f"--train_data_dir={TRAIN_DATA_DIR}",
    f"--output_dir={OUTPUT_DIR}",
    f"--output_name={OUTPUT_NAME}",
    "--network_module=networks.lora",
    "--network_dim=8",
    "--network_alpha=4",
    "--resolution=768,768",
    "--enable_bucket",
    "--min_bucket_reso=512",
    "--max_bucket_reso=1024",
    "--train_batch_size=1",
    "--max_train_steps=3000",
    "--learning_rate=1e-4",
    "--optimizer_type=AdamW",
    "--lr_scheduler=cosine",
    "--mixed_precision=fp16",
    "--save_precision=fp16",
    "--cache_latents",
    "--cache_latents_to_disk",
    "--cache_text_encoder_outputs",
    "--cache_text_encoder_outputs_to_disk",
    "--gradient_checkpointing",
    "--no_half_vae",
    "--max_data_loader_n_workers=0",
    "--network_train_unet_only",
    "--caption_extension=.txt",
    "--save_model_as=safetensors",
    "--save_every_n_steps=200",
    "--save_last_n_steps_state=1",
    "--save_state",
    "--seed=42",
)
if resume_state is not None:
    train_command += (f"--resume={resume_state}",)
run("train", *train_command)

Publish the Notebook output with **Save Version** after training. To resume in
a later session, add that Notebook output as an Input, then run this notebook
again. Kaggle inputs are read-only; only `/kaggle/working` becomes an output.